# 08 — Self-improvement loop (real data)

Drives the three learning layers using **real** RAG output (no `chunk-x` /
`chunk-y` placeholders):

1. **Chunk-quality scores** — re-rank using actual chunk IDs returned by the search index.
2. **Learned rules** — distilled from real corrections on real answers.
3. **Golden Q&A pairs** — promoted from real 👍 turns.

**Prerequisites**
- Run `06_ingestion_pipeline.ipynb` so at least one document is indexed.
- Run `07_rag_query.ipynb` once to confirm retrieval works.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.search_client import SearchService
from src.openai_client import OpenAIService
from src.cosmos_client import create_state_service
from src.rag import RAGEngine
from src.learning import LearningLoop
from src.models import FeedbackRecord, FeedbackChunkMeta

cosmos = create_state_service(); cosmos.ensure_containers()
ai = OpenAIService()
rag = RAGEngine(SearchService(), ai, cosmos)
learner = LearningLoop(cosmos, ai)

SESSION_ID = 'nb-learn-real'
USER_ID = 'user-input'
session_id='nb-rag-session'
user_id='user-input'
print('State backend:', type(cosmos).__name__)


INFO azure.identity._credentials.environment: Incomplete environment configuration for EnvironmentCredential. These variables are set: AZURE_TENANT_ID
INFO azure.identity._credentials.managed_identity: ManagedIdentityCredential will use IMDS
WARNING src.cosmos_client: COSMOS_ENDPOINT/COSMOS_KEY not set - using LocalStateService
WARNING src.local_state: Using LocalStateService (file-backed) at \app\data\state.json — Cosmos DB is NOT configured
INFO src.local_state: LocalStateService ready: ['sessions', 'documents', 'feedback', 'learned_rules', 'golden_pairs', 'chunk_quality', 'ingestion_tasks']


State backend: LocalStateService


In [3]:
# removing all the existing feedback for a clean slate
cosmos.wipe_all()

{'sessions': 0,
 'documents': 0,
 'feedback': 0,
 'learned_rules': 0,
 'golden_pairs': 0,
 'chunk_quality': 0,
 'ingestion_tasks': 0}

In [4]:
# 1. Ask real questions against the index, then submit real feedback.
#    Tune QUESTIONS to match content in YOUR indexed documents.
#    Each entry is (question, rating, correction_or_None):
#       rating='down' + correction → trains rules + penalizes cited chunks
#       rating='up'                → promotes to golden pair + boosts cited chunks
QUESTIONS = [
    ('What is the architecture of multi agent research?',  'up',   None),
    ('Summarize the key findings of the document.',        'up',   None),
    ('How many agents are described in the system?',       'down', 'Be precise with numbers — quote the exact figure from the document instead of approximating.'),
    ('What evaluation metrics are reported?',              'down', 'Always cite the page number when listing metrics.'),
    ('Describe the diagrams included in the paper.',       'down', 'Do not describe images that are merely decorative; focus on architecture diagrams only.'),
]

turns = []
for q, rating, correction in QUESTIONS:
    answer, sources, turn = rag.answer(q, session_id=SESSION_ID, user_id=USER_ID)
    chunk_ids  = [s.chunk_id for s in sources]
    chunk_meta = [FeedbackChunkMeta(chunk_id=s.chunk_id, type=s.type, page=s.page) for s in sources]

    cosmos.save_feedback(FeedbackRecord(
        session_id=SESSION_ID, turn_id=turn.id, user_id=USER_ID,
        rating=rating, correction=correction,
        question=q, answer=answer,
        chunk_ids=chunk_ids, chunk_meta=chunk_meta,
    ))
    turns.append({'q': q, 'rating': rating, 'chunk_ids': chunk_ids})
    print(f"[{rating:>4}] {q}")
    print(f"   → {len(chunk_ids)} chunks: {chunk_ids[:3]}{'…' if len(chunk_ids) > 3 else ''}")
    print(f"   → answer: {answer[:120].replace(chr(10), ' ')}…\n")

print(f'persisted {len(turns)} real feedback records ✔')


INFO httpx: HTTP Request: POST https://azr-oai-dai-sandbox4-001.openai.azure.com/openai/deployments/text-embedding-ada-002/embeddings?api-version=2024-08-01-preview "HTTP/1.1 200 OK"
INFO azure.core.pipeline.policies.http_logging_policy: Request URL: 'https://agent-ai-search5yc2.search.windows.net/indexes('pdg-was-multimodal-rag-2')/docs/search.post.search?api-version=2024-05-01-preview'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '34780'
    'api-key': 'REDACTED'
    'Accept': 'application/json;odata.metadata=none'
    'x-ms-client-request-id': '7f7acdb1-4b87-11f1-aa95-bcd22c162bea'
    'User-Agent': 'azsdk-python-search-documents/11.6.0b4 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
A body is sent with the request
INFO azure.core.pipeline.policies.http_logging_policy: Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
    'Content-Type': 'application/json; odata.metadata=none; odata.streaming=true; charset=

[  up] What is the architecture of multi agent research?
   → 5 chunks: ['d8654259-7552-48b8-9317-9adf8aa5eb59', 'efef6694-09a9-489d-9d03-55d4a37db27c', '6d2c05c6-35a6-4610-ae27-093e2d5791be']…
   → answer: The architecture of the Multi-Agent Research System is based on a graph-based orchestration pattern. Each stage of the r…



INFO httpx: HTTP Request: POST https://azr-oai-dai-sandbox4-001.openai.azure.com/openai/deployments/text-embedding-ada-002/embeddings?api-version=2024-08-01-preview "HTTP/1.1 200 OK"
INFO azure.core.pipeline.policies.http_logging_policy: Request URL: 'https://agent-ai-search5yc2.search.windows.net/indexes('pdg-was-multimodal-rag-2')/docs/search.post.search?api-version=2024-05-01-preview'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '34756'
    'api-key': 'REDACTED'
    'Accept': 'application/json;odata.metadata=none'
    'x-ms-client-request-id': '8209b2fd-4b87-11f1-9372-bcd22c162bea'
    'User-Agent': 'azsdk-python-search-documents/11.6.0b4 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
A body is sent with the request
INFO azure.core.pipeline.policies.http_logging_policy: Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
    'Content-Type': 'application/json; odata.metadata=none; odata.streaming=true; charset=

[  up] Summarize the key findings of the document.
   → 5 chunks: ['9edd1c7e-6f36-47a2-9d8f-c3295f846771', 'ef0c2136-b4d0-4cf8-9d27-42363164de73', '6d2c05c6-35a6-4610-ae27-093e2d5791be']…
   → answer: The document outlines the architecture and capabilities of the Multi-Agent Research System, highlighting its agentic arc…



INFO httpx: HTTP Request: POST https://azr-oai-dai-sandbox4-001.openai.azure.com/openai/deployments/text-embedding-ada-002/embeddings?api-version=2024-08-01-preview "HTTP/1.1 200 OK"
INFO azure.core.pipeline.policies.http_logging_policy: Request URL: 'https://agent-ai-search5yc2.search.windows.net/indexes('pdg-was-multimodal-rag-2')/docs/search.post.search?api-version=2024-05-01-preview'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '34841'
    'api-key': 'REDACTED'
    'Accept': 'application/json;odata.metadata=none'
    'x-ms-client-request-id': '842e3018-4b87-11f1-9056-bcd22c162bea'
    'User-Agent': 'azsdk-python-search-documents/11.6.0b4 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
A body is sent with the request
INFO azure.core.pipeline.policies.http_logging_policy: Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
    'Content-Type': 'application/json; odata.metadata=none; odata.streaming=true; charset=

[down] How many agents are described in the system?
   → 5 chunks: ['76768357-fa30-476b-9218-813840b4f1f1', '6d2c05c6-35a6-4610-ae27-093e2d5791be', '64d590fe-6400-4e2c-8355-2d35bfd23023']…
   → answer: The system allows for the deployment of 10 or more sub-agents in parallel for complex deep-research queries. The number …



INFO httpx: HTTP Request: POST https://azr-oai-dai-sandbox4-001.openai.azure.com/openai/deployments/text-embedding-ada-002/embeddings?api-version=2024-08-01-preview "HTTP/1.1 200 OK"
INFO azure.core.pipeline.policies.http_logging_policy: Request URL: 'https://agent-ai-search5yc2.search.windows.net/indexes('pdg-was-multimodal-rag-2')/docs/search.post.search?api-version=2024-05-01-preview'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '34761'
    'api-key': 'REDACTED'
    'Accept': 'application/json;odata.metadata=none'
    'x-ms-client-request-id': '861e8d30-4b87-11f1-afc9-bcd22c162bea'
    'User-Agent': 'azsdk-python-search-documents/11.6.0b4 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
A body is sent with the request
INFO azure.core.pipeline.policies.http_logging_policy: Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
    'Content-Type': 'application/json; odata.metadata=none; odata.streaming=true; charset=

[down] What evaluation metrics are reported?
   → 5 chunks: ['93938a9a-8afa-4b32-88d6-d7e868e3c76c', '8334add9-1bd6-4899-8cac-fd3bd2ff0019', '70d1d52c-ca30-4081-a5e6-b967df91f050']…
   → answer: The evaluation metrics reported for the Multi-Agent Research System include time savings, cost reduction, and quality im…



INFO httpx: HTTP Request: POST https://azr-oai-dai-sandbox4-001.openai.azure.com/openai/deployments/text-embedding-ada-002/embeddings?api-version=2024-08-01-preview "HTTP/1.1 200 OK"
INFO azure.core.pipeline.policies.http_logging_policy: Request URL: 'https://agent-ai-search5yc2.search.windows.net/indexes('pdg-was-multimodal-rag-2')/docs/search.post.search?api-version=2024-05-01-preview'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '34804'
    'api-key': 'REDACTED'
    'Accept': 'application/json;odata.metadata=none'
    'x-ms-client-request-id': '884238ec-4b87-11f1-a190-bcd22c162bea'
    'User-Agent': 'azsdk-python-search-documents/11.6.0b4 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
A body is sent with the request
INFO azure.core.pipeline.policies.http_logging_policy: Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
    'Content-Type': 'application/json; odata.metadata=none; odata.streaming=true; charset=

[down] Describe the diagrams included in the paper.
   → 5 chunks: ['2fefc443-430a-475f-8196-a0b95af3de1d', '58013261-71a9-4ba5-af42-4abc00b38964', '2f218a76-5122-4884-98ab-9331875e3126']…
   → answer: The paper includes several diagrams:  1. **Sequence Diagram (page 6)**: This diagram illustrates the interaction between…

persisted 5 real feedback records ✔


In [5]:
cosmos.list_feedback()


[FeedbackRecord(id='3fc2241a-8c0d-463f-85ff-0e78f8526d21', session_id='nb-learn-real', turn_id='9a333b24-2400-4f43-a6ac-7f2f47417ee8', user_id='user-input', rating='down', correction='Do not describe images that are merely decorative; focus on architecture diagrams only.', question='Describe the diagrams included in the paper.', answer="The paper includes several diagrams:\n\n1. **Sequence Diagram (page 6)**: This diagram illustrates the interaction between different components in a system, detailing how they communicate and process information.\n\n2. **Flowchart Diagram (page 4)**: This diagram shows the process of building edges in a graph, outlining the components and their relationships in a structured manner.\n\n3. **Architecture Flow Diagram (page 15)**: This diagram depicts the complete end-to-end flow of the Multi-Agent Research System, including nodes, edges, feedback loops, and conditional branching paths.\n\n4. **Flow Diagram using Azure (page 6)**: This diagram illustrates 

In [17]:
cosmos.list_documents(user_id=USER_ID)

[]

In [8]:
cosmos.get_chunk_quality(chunk_id="26a80744-9610-40ac-9dd9-b36093ba36f1")

In [7]:
stats = learner.run_once()
print('Learning stats:', stats)

DEBUG src.learning: [run_once] starting learning loop
DEBUG src.learning: [run_once] fetched 5 feedback records (up=2, down=3)
DEBUG src.learning: [run_once] -> Layer 2: chunk-quality update
DEBUG src.learning: [chunk_quality] fb=3fc2241a-8c0d-463f-85ff-0e78f8526d21 rating=down chunk_ids=5 correction='Do not describe images that are merely decorative; focus on architecture diagrams only.'
DEBUG src.learning: [chunk_quality] +bad chunk=2fefc443-430a-475f-8196-a0b95af3de1d (fb=3fc2241a-8c0d-463f-85ff-0e78f8526d21)
DEBUG src.learning: [chunk_quality] +bad chunk=58013261-71a9-4ba5-af42-4abc00b38964 (fb=3fc2241a-8c0d-463f-85ff-0e78f8526d21)
DEBUG src.learning: [chunk_quality] +bad chunk=2f218a76-5122-4884-98ab-9331875e3126 (fb=3fc2241a-8c0d-463f-85ff-0e78f8526d21)
DEBUG src.learning: [chunk_quality] +bad chunk=1a379a90-d5b4-4413-bf3e-e51db0c21524 (fb=3fc2241a-8c0d-463f-85ff-0e78f8526d21)
DEBUG src.learning: [chunk_quality] +bad chunk=101a73c7-576d-4362-8c87-3779ca1b1f40 (fb=3fc2241a-8c0d-46

Learning stats: {'feedback_count': 5, 'rules_added': 3, 'golden_added': 2, 'chunk_updates': 20}


In [ ]:
# 2. Run all three learning layers.
stats = learner.run_once()
print('Learning stats:', stats)

print('\nRules learned:')
for r in cosmos.get_rules():
    print(' •', r.rule)

print('\nGolden pairs:')
for g in cosmos.get_golden_pairs():
    print(f'  Q: {g.question}\n     → {g.answer[:120]}…')


In [ ]:
# 3. Verify chunk-quality on the real chunks that were cited.
seen = set()
for t in turns:
    print(f"\nFeedback for question: {t['q']}")
    print(f"Rating: {t['rating']}")
    print(f"Cited chunks: {t['chunk_ids']}")

    label = '👍' if t['rating'] == 'up' else '👎'
    for cid in t['chunk_ids']:
        if cid in seen:
            continue
        seen.add(cid)
        q = cosmos.get_chunk_quality(cid)
        if q is None:
            print(f'  {label} {cid}: (no quality record yet)')
        else:
            print(f'  {label} {cid}: good={q.times_in_good_answer}  bad={q.times_in_bad_answer}')



Feedback for question: What is the architecture of multi agent research?
Rating: up
Cited chunks: ['d8654259-7552-48b8-9317-9adf8aa5eb59', 'efef6694-09a9-489d-9d03-55d4a37db27c', '6d2c05c6-35a6-4610-ae27-093e2d5791be', '8f87eb89-68f7-46ba-be07-da196148850a', '2f218a76-5122-4884-98ab-9331875e3126']
Chunk quality:
  👍 d8654259-7552-48b8-9317-9adf8aa5eb59: good=2  bad=0
  👍 efef6694-09a9-489d-9d03-55d4a37db27c: good=1  bad=0
  👍 6d2c05c6-35a6-4610-ae27-093e2d5791be: good=2  bad=1
  👍 8f87eb89-68f7-46ba-be07-da196148850a: good=1  bad=0
  👍 2f218a76-5122-4884-98ab-9331875e3126: good=1  bad=1

Feedback for question: Summarize the key findings of the document.
Rating: up
Cited chunks: ['9edd1c7e-6f36-47a2-9d8f-c3295f846771', 'ef0c2136-b4d0-4cf8-9d27-42363164de73', '6d2c05c6-35a6-4610-ae27-093e2d5791be', 'd76a5528-a233-4bd8-8bb8-1c2bb5ba7ba6', 'd8654259-7552-48b8-9317-9adf8aa5eb59']
Chunk quality:
  👍 9edd1c7e-6f36-47a2-9d8f-c3295f846771: good=1  bad=0
  👍 ef0c2136-b4d0-4cf8-9d27-42363164de73

In [15]:
cosmos.get_chunk_quality('2fefc443-430a-475f-8196-a0b95af3de1d')

ChunkQuality(id='2fefc443-430a-475f-8196-a0b95af3de1d', chunk_id='2fefc443-430a-475f-8196-a0b95af3de1d', times_retrieved=0, times_in_good_answer=0, times_in_bad_answer=1, quality_score=0.0, updated_at='2026-05-09T09:15:03.957584+00:00')

In [ ]:
# 4. Sanity check — re-ask one of the corrected questions in a fresh
#    session. The system prompt now includes the distilled rules +
#    golden pairs, so the new answer should reflect them.
followup = next(q for q, r, _ in QUESTIONS if r == 'down')
new_answer, new_sources, _ = rag.answer(followup, session_id=SESSION_ID + '-after', user_id=USER_ID)
print('Q:', followup)
print('A:', new_answer)
print('Chunk quality:', [cosmos.get_chunk_quality(s.chunk_id) for s in new_sources])
print('\nsources:', [(s.chunk_id, s.page, s.type) for s in new_sources])


INFO httpx: HTTP Request: POST https://azr-oai-dai-sandbox4-001.openai.azure.com/openai/deployments/text-embedding-ada-002/embeddings?api-version=2024-08-01-preview "HTTP/1.1 200 OK"
INFO azure.core.pipeline.policies.http_logging_policy: Request URL: 'https://agent-ai-search5yc2.search.windows.net/indexes('pdg-was-multimodal-rag-2')/docs/search.post.search?api-version=2024-05-01-preview'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '34841'
    'api-key': 'REDACTED'
    'Accept': 'application/json;odata.metadata=none'
    'x-ms-client-request-id': '3b31858a-4b88-11f1-a028-bcd22c162bea'
    'User-Agent': 'azsdk-python-search-documents/11.6.0b4 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
A body is sent with the request
INFO azure.core.pipeline.policies.http_logging_policy: Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
    'Content-Type': 'application/json; odata.metadata=none; odata.streaming=true; charset=

Q: How many agents are described in the system?
A: The document does not specify the exact number of agents described in the system.

sources: [('6d2c05c6-35a6-4610-ae27-093e2d5791be', 27, 'text')]
